In [4]:
import os
import gradio as gr
from dotenv import load_dotenv
from groq import Groq

# 1. Initialize Client
load_dotenv()
client = Groq(api_key=os.environ.get("GROQ_API_KEY"))

car_expert_persona = """
You are 'Car World AI', an expert automotive consultant. u are developed by ansh anand not open ai
Provide clear, professional breakdowns of vehicle reliability, specs, and recommendations.
Maintain an enthusiastic, helpful, and knowledgeable tone.
Always review past questions to handle follow-up queries accurately dont give up long paragarph answer if user ask in 3-4 words explain in 5-6 lines and if ask for details explain in details using tables differentiate etc.
"""

# GLOBAL BACKEND RECOVERY SHIELD: Acts as the true memory storage
# Bypasses all changing Gradio data structure formats entirely
CUSTOM_CHAT_HISTORY = []

def respond(user_message, chat_history):
    global CUSTOM_CHAT_HISTORY
    
    if not user_message.strip():
        return "", chat_history

    # Start the message payload stack safely
    api_messages = [{"role": "system", "content": str(car_expert_persona)}]
    
    # Process history using our ultra-stable backend global variable list
    for user_turn, bot_turn in CUSTOM_CHAT_HISTORY:
        if user_turn and bot_turn:
            api_messages.append({"role": "user", "content": str(user_turn).strip()})
            api_messages.append({"role": "assistant", "content": str(bot_turn).strip()})
            
    # Append the incoming active query
    api_messages.append({"role": "user", "content": str(user_message).strip()})
    
    try:
        # Call Groq's high-speed engine
        completion = client.chat.completions.create(
            model="openai/gpt-oss-120b",
            messages=api_messages,
            temperature=0.7,
            stream=False 
        )
        
        # FIXED: Added [0] index to correctly extract the content from the list array object
        bot_response = completion.choices[0].message.content
        
    except Exception as e:
        bot_response = f"⚠️ Groq Interface Error: {str(e)}"
        
    # Append the text to our clean, controlled backup state memory list
    CUSTOM_CHAT_HISTORY.append((user_message, bot_response))
    
    # Update Gradio's visual UI interface array automatically based on incoming component shapes
    try:
        chat_history.append({"role": "user", "content": str(user_message)})
        chat_history.append({"role": "assistant", "content": str(bot_response)})
    except Exception:
        chat_history.append([user_message, bot_response])
    
    # Return empty string to clear the text input box, and update history
    return "", chat_history

def clear_session():
    global CUSTOM_CHAT_HISTORY
    CUSTOM_CHAT_HISTORY = []
    return None

# 3. Build the Clean Blocks Layout
with gr.Blocks(title="🚗 Car World AI") as demo:
    gr.Markdown("# 🚗 Car World AI Chatbot")
    
    chatbot = gr.Chatbot(label="Conversation")
    
    with gr.Row():
        txt_input = gr.Textbox(
            show_label=False, 
            placeholder="Type your car question here (e.g., 'show me a car')...",
            container=False
        )
        submit_btn = gr.Button("Send", variant="primary")
        
    clear_btn = gr.Button("Clear Conversation")

    # Wire up standard pipeline logic
    txt_input.submit(respond, [txt_input, chatbot], [txt_input, chatbot])
    submit_btn.click(respond, [txt_input, chatbot], [txt_input, chatbot])
    clear_btn.click(clear_session, None, chatbot, queue=False)

# Launch local app server
if __name__ == "__main__":
    demo.queue().launch(share=False)


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
